# Nettoyage de la base d'apprentissage

In [46]:
import numpy as np
import pandas as pd
import sys
import os

# On connecte le notebook à tous les fichiers inclus dans le dossier /fonctions
sys.path.append(os.path.abspath("../fonctions"))

%load_ext autoreload
%autoreload 2

from cleaning import *

# On charge la base d'apprentissage
df = pd.read_csv("../data_finale/base_apprentissage.csv")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Traitement des doublons

In [47]:
verifier_doublons_metier_et_techniques(df)

Recherche de doublons
 ATTENTION : 787 lignes sont des doublons techniques stricts.
   (Même joueur, même saison, même club -> Erreur d'extraction/jointure)

   Exemple de lignes techniques concernées :
                player  season         team
28             Willian    2021      Arsenal
29             Willian    2021      Arsenal
30             Willian    2021      Arsenal
37   Emiliano Martínez    2021  Aston Villa
38   Emiliano Martínez    2021  Aston Villa
119           Jorginho    2021      Chelsea
=869 lignes correspondent à des doublons de mercato
   (Même joueur, même saison, mais CLUBS DIFFÉRENTS -> Transferts de mi-saison)

   Exemple de joueurs transférés concernés :
                    player  season     team
0   Ainsley Maitland-Niles    2021  Arsenal
14             Joe Willock    2021  Arsenal
16         Martin Ødegaard    2021  Arsenal
17             Mathew Ryan    2021  Arsenal
25          Sead Kolašinac    2021  Arsenal
26        Shkodran Mustafi    2021  Arsenal


{'doublons_techniques': np.int64(787), 'doublons_mercato': np.int64(869)}

In [48]:
df = fusionner_doublons_techniques(df)

Format initial de la base : (17122, 129)
Format après fusion intelligente des doublons : (16335, 129)


In [49]:
verifier_doublons_mercato(df)

Diagnostic des doublons liés au mercato (transferts de mi-saison)
Attention : 869 lignes sont des doublons pour le même joueur lors de la même saison !
   Cela indique la présence de transferts ou de prêts à la mi-saison.

   Exemple de lignes concernées :
             player  season        team
40     Aarón Martín    2021  Celta Vigo
41     Aarón Martín    2021    Mainz 05
49     Abakar Sylla    2526      Nantes
50     Abakar Sylla    2526  Strasbourg
58  Abde Ezzalzouli    2324   Barcelona
59  Abde Ezzalzouli    2324  Real Betis


In [51]:
df = fusionner_et_recalculer_mercato(df)

Format avant fusion mercato : (16335, 129)
Format après fusion mercato : (15466, 129)
Recalcul des ratios et statistiques par 90 minutes...
Base de données fusionnée et variables recalculées avec exactitude.



<string>:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
<string>:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`


In [ ]:
df

In [225]:
# A FAIRE APRES LA GESTION DES DOUBLONS MERCATOS

# df = df[df["Playing Time_Min"] > (5 * 90)]

## Traitement des variables avec beaucoup de valeurs manquantes

In [213]:
diagnostiquer_valeurs_manquantes(df, seuil=0.30)

Diagnostic des valeurs manquantes (Seuil > 30%)
   • Performance_PKwon : 100.0% de valeurs manquantes
   • Performance_PKcon : 100.0% de valeurs manquantes
   • Penalty Kicks_Save% : 94.8% de valeurs manquantes
   • Performance_CS% : 93.1% de valeurs manquantes
   • Performance_Save% : 92.9% de valeurs manquantes
   • Performance_GA : 92.7% de valeurs manquantes
   • Performance_W : 92.7% de valeurs manquantes
   • Performance_Saves : 92.7% de valeurs manquantes
   • Performance_SoTA : 92.7% de valeurs manquantes
   • Performance_GA90 : 92.7% de valeurs manquantes
   • Penalty Kicks_PKatt : 92.7% de valeurs manquantes
   • Performance_D : 92.7% de valeurs manquantes
   • Performance_L : 92.7% de valeurs manquantes
   • Performance_CS : 92.7% de valeurs manquantes
   • Penalty Kicks_PKm : 92.7% de valeurs manquantes
   • Penalty Kicks_PKsv : 92.7% de valeurs manquantes
   • Penalty Kicks_PKA : 92.7% de valeurs manquantes
   • xg : 49.1% de valeurs manquantes
   • xa : 49.1% de valeurs m

['Performance_PKwon',
 'Performance_PKcon',
 'Penalty Kicks_Save%',
 'Performance_CS%',
 'Performance_Save%',
 'Performance_GA',
 'Performance_W',
 'Performance_Saves',
 'Performance_SoTA',
 'Performance_GA90',
 'Penalty Kicks_PKatt',
 'Performance_D',
 'Performance_L',
 'Performance_CS',
 'Penalty Kicks_PKm',
 'Penalty Kicks_PKsv',
 'Penalty Kicks_PKA',
 'xg',
 'xa',
 'np_xg',
 'xg_chain',
 'xg_buildup']

On sépare les joueurs de champs des gardiens car on remarque que les variables ayant un fort taux de valeurs manquantes concerne les gardiens.

In [214]:
# On sépare les gardiens des joueurs de champ

# On s'assure que le poste 'pos' est bien au format texte pour filtrer
df["pos"] = df["pos"].astype(str)

# Uniquement les Gardiens de but (GK)
df_gardiens = df[df["pos"].str.contains("GK|Gardien", na=False)].copy()

# Tous les autres joueurs de champ (Attaquants, Milieux, Défenseurs)
df_champs = df[~df["pos"].str.contains("GK|Gardien", na=False)].copy()

In [215]:
diagnostiquer_valeurs_manquantes(df_gardiens, seuil=0.30)

Diagnostic des valeurs manquantes (Seuil > 30%)
   • Performance_PKwon : 100.0% de valeurs manquantes
   • Performance_PKcon : 100.0% de valeurs manquantes
   • Standard_G/SoT : 98.9% de valeurs manquantes
   • Standard_G/Sh : 96.0% de valeurs manquantes
   • Standard_SoT% : 96.0% de valeurs manquantes
   • Subs_Mn/Sub : 83.0% de valeurs manquantes
   • xg : 49.9% de valeurs manquantes
   • xa : 49.9% de valeurs manquantes
   • np_xg : 49.9% de valeurs manquantes
   • xg_chain : 49.9% de valeurs manquantes
   • xg_buildup : 49.9% de valeurs manquantes

Total : 11 colonnes dépassent le seuil de 30%.


['Performance_PKwon',
 'Performance_PKcon',
 'Standard_G/SoT',
 'Standard_G/Sh',
 'Standard_SoT%',
 'Subs_Mn/Sub',
 'xg',
 'xa',
 'np_xg',
 'xg_chain',
 'xg_buildup']

In [216]:
diagnostiquer_valeurs_manquantes(df_champs, seuil=0.30)

Diagnostic des valeurs manquantes (Seuil > 30%)
   • Performance_Save% : 100.0% de valeurs manquantes
   • Performance_PKwon : 100.0% de valeurs manquantes
   • Performance_PKcon : 100.0% de valeurs manquantes
   • Penalty Kicks_Save% : 100.0% de valeurs manquantes
   • Performance_GA90 : 100.0% de valeurs manquantes
   • Performance_GA : 100.0% de valeurs manquantes
   • Performance_W : 100.0% de valeurs manquantes
   • Performance_Saves : 100.0% de valeurs manquantes
   • Performance_SoTA : 100.0% de valeurs manquantes
   • Performance_D : 100.0% de valeurs manquantes
   • Penalty Kicks_PKatt : 100.0% de valeurs manquantes
   • Performance_L : 100.0% de valeurs manquantes
   • Performance_CS : 100.0% de valeurs manquantes
   • Performance_CS% : 100.0% de valeurs manquantes
   • Penalty Kicks_PKm : 100.0% de valeurs manquantes
   • Penalty Kicks_PKsv : 100.0% de valeurs manquantes
   • Penalty Kicks_PKA : 100.0% de valeurs manquantes
   • xg : 49.0% de valeurs manquantes
   • xa : 49.

['Performance_Save%',
 'Performance_PKwon',
 'Performance_PKcon',
 'Penalty Kicks_Save%',
 'Performance_GA90',
 'Performance_GA',
 'Performance_W',
 'Performance_Saves',
 'Performance_SoTA',
 'Performance_D',
 'Penalty Kicks_PKatt',
 'Performance_L',
 'Performance_CS',
 'Performance_CS%',
 'Penalty Kicks_PKm',
 'Penalty Kicks_PKsv',
 'Penalty Kicks_PKA',
 'xg',
 'xa',
 'np_xg',
 'xg_chain',
 'xg_buildup']

In [217]:
df_gardiens = nettoyer_colonnes_vides(df_gardiens)

Nettoyage des valeurs manquantes (Seuil > 80%)
Colonnes avec plus de 80% de valeurs manquantes :
   • Performance_PKwon : 100.0% de valeurs manquantes
   • Performance_PKcon : 100.0% de valeurs manquantes
   • Standard_G/SoT : 98.9% de valeurs manquantes
   • Standard_SoT% : 96.0% de valeurs manquantes
   • Standard_G/Sh : 96.0% de valeurs manquantes
   • Subs_Mn/Sub : 83.0% de valeurs manquantes

Succès : 6 colonnes ont été supprimées du dataset.


In [218]:
df_champs = nettoyer_colonnes_vides(df_champs)

Nettoyage des valeurs manquantes (Seuil > 80%)
Colonnes avec plus de 80% de valeurs manquantes :
   • Performance_Save% : 100.0% de valeurs manquantes
   • Performance_PKcon : 100.0% de valeurs manquantes
   • Penalty Kicks_Save% : 100.0% de valeurs manquantes
   • Performance_PKwon : 100.0% de valeurs manquantes
   • Performance_GA : 100.0% de valeurs manquantes
   • Performance_W : 100.0% de valeurs manquantes
   • Performance_GA90 : 100.0% de valeurs manquantes
   • Performance_SoTA : 100.0% de valeurs manquantes
   • Performance_Saves : 100.0% de valeurs manquantes
   • Performance_CS : 100.0% de valeurs manquantes
   • Performance_L : 100.0% de valeurs manquantes
   • Performance_D : 100.0% de valeurs manquantes
   • Performance_CS% : 100.0% de valeurs manquantes
   • Penalty Kicks_PKsv : 100.0% de valeurs manquantes
   • Penalty Kicks_PKA : 100.0% de valeurs manquantes
   • Penalty Kicks_PKatt : 100.0% de valeurs manquantes
   • Penalty Kicks_PKm : 100.0% de valeurs manquantes

S

## Encodage de variables

In [219]:
# Analyser la base des gardiens
cols_gardiens = lister_variables_categorielles(df_gardiens)

Variables catégorielles du dataset :
Liste des variables catégorielles détectées :
   • league (5 modalités uniques)
   • team (136 modalités uniques)
   • player (430 modalités uniques)
   • nation (65 modalités uniques)
   • pos (1 modalités uniques)
   • age (179 modalités uniques)
   • join_key (430 modalités uniques)
   • match_method (5 modalités uniques)
   • date (95 modalités uniques)
   • date_of_birth (353 modalités uniques)
   • name (358 modalités uniques)
   • tm_join_key (358 modalités uniques)
   • tm_join_key_full (358 modalités uniques)

Total : 13 variables catégorielles trouvées.


In [220]:
# Analyser la base des joueurs de champ
cols_champs = lister_variables_categorielles(df_champs)

Variables catégorielles du dataset :
Liste des variables catégorielles détectées :
   • league (5 modalités uniques)
   • team (137 modalités uniques)
   • player (5086 modalités uniques)
   • nation (129 modalités uniques)
   • pos (9 modalités uniques)
   • age (1653 modalités uniques)
   • join_key (5085 modalités uniques)
   • match_method (15 modalités uniques)
   • date (244 modalités uniques)
   • date_of_birth (3067 modalités uniques)
   • name (4194 modalités uniques)
   • tm_join_key (4193 modalités uniques)
   • tm_join_key_full (4193 modalités uniques)

Total : 13 variables catégorielles trouvées.


In [221]:
# Traitement de la base des gardiens
df_gardiens_pret = encoder_dataset_football(df_gardiens)

# Traitement de la base des joueurs de champ
df_champs_pret = encoder_dataset_football(df_champs)

Format initial avant encodage : (1189, 123)
Profil Gardien détecté : Suppression de la colonne 'pos'.
Colonne 'nation' regroupée : Top 10 + 'Autre' (11 modalités au total).
Format final après encodage : (1189, 136)

Format initial avant encodage : (15146, 112)
Profil Joueurs de champ détecté : Encodage Multi-Label des 9 modalités de postes.
   • Encodage des postes terminé. Classes détectées : ['DF', 'FW', 'MF']
Colonne 'nation' regroupée : Top 10 + 'Autre' (11 modalités au total).
Format final après encodage : (15146, 128)



## Sauvegarde des bases d'apprentissage mises à jour

In [222]:
df_gardiens_pret.to_csv(r'..\data_finale\base_gardiens.csv', index=False, sep=',', encoding='utf-8-sig')
df_champs_pret.to_csv(r'..\data_finale\base_joueurs_champs.csv', index=False, sep=',', encoding='utf-8-sig')

In [ ]:
df_champs_pret